In [ ]:
import torch

from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, IterableDataset

import json
from pyarrow import parquet as pq

In [ ]:
dataset = pq.ParquetDataset("../data/main/spa-eng/train_spa_eng/")

In [ ]:
class LoadLanguageData(IterableDataset):
    
    def __init__(self, path:str, columns:list) -> None:

        super().__init__()

        self.cols = columns
        self.path = path
        self.dataset = pq.ParquetDataset(self.path)

    def __iter__(self):

        for fragment in self.dataset.fragments:

            table = fragment.to_table()
            rows = table.to_pylist()

            for row in rows:

                yield {
                    'input_ids' : torch.tensor(row[self.cols[1]], dtype=torch.long),
                    'output_ids': torch.tensor(row[self.cols[2]], dtype=torch.long),
                    'target_ids': torch.tensor(row[self.cols[3]], dtype=torch.long)
                }

def wrapper_collate_batch(pad_value_in, pad_value_out):

    def collate_batch(batch):

        input_list  = [item['input_ids']  for item in batch]
        output_list = [item['output_ids'] for item in batch]
        target_list = [item['target_ids'] for item in batch]

        input_padded  = pad_sequence(input_list,  batch_first=True, padding_value=pad_value_in)
        output_padded = pad_sequence(output_list, batch_first=True, padding_value=pad_value_out)
        target_padded = pad_sequence(target_list, batch_first=True, padding_value=pad_value_out)

        return (input_padded, output_padded, target_padded)

    return collate_batch


def load_tokenizer(path:str) -> tuple:

    with open(path) as f:
        tokenizer = json.load(f)
    
    word2index = tokenizer["vocab"]
    special_tokens = tokenizer["special_tokens"]

    return special_tokens, word2index


def index2word(input_tensor: torch.Tensor, vocab_mapper: dict, unk_token:int) -> list:

    indices_matrix = input_tensor.cpu().numpy()

    output = [
        [vocab_mapper.get(idx, vocab_mapper[unk_token]) for idx in sentence]
        for sentence in indices_matrix
    ]
    
    return output

In [ ]:
sp_tokens_eng, vocab_idx2word_eng = load_tokenizer("../data/main/vocabs/eng_tokenizer.json")
sp_tokens_spa, vocab_idx2word_spa = load_tokenizer("../data/main/vocabs/spa_tokenizer.json")

pad_value_eng = sp_tokens_eng["[PAD]"]
pad_value_spa = sp_tokens_spa["[PAD]"]

In [ ]:
train_path = "../data/main/spa-eng/train_spa_eng"
train_schema = pq.ParquetDataset(train_path)
train_cols = train_schema.schema.names

test_path = "../data/main/spa-eng/test_spa_eng"
test_schema = pq.ParquetDataset(test_path)
test_cols = test_schema.schema.names

In [ ]:
train_data = LoadLanguageData(train_path, train_cols)
train_loader = DataLoader(train_data, batch_size=64, collate_fn=wrapper_collate_batch(pad_value_eng, pad_value_spa))

test_data = LoadLanguageData(test_path, test_cols)
test_loader = DataLoader(test_data, batch_size=64, collate_fn=wrapper_collate_batch(pad_value_eng, pad_value_spa))

In [ ]:
#for eng_in, spa_out, spa_t in train_loader:
#    break

#print(np.array(index2word(eng_in, vocab_idx2word_eng, vocab_word2idx_eng["[UNK]"]))[:3], end="\n\n\n")
#print(np.array(index2word(spa_out, vocab_idx2word_spa, vocab_word2idx_spa["[UNK]"]))[:3], end="\n\n\n")
#print(np.array(index2word(spa_t, vocab_idx2word_spa, vocab_word2idx_spa["[UNK]"]))[:3])

In [ ]:
#for eng_in, spa_out, spa_t in test_loader:
#    break

#print(np.array(index2word(eng_in, vocab_idx2word_eng, vocab_word2idx_eng["[UNK]"]))[:3], end="\n\n\n")
#print(np.array(index2word(spa_out, vocab_idx2word_spa, vocab_word2idx_spa["[UNK]"]))[:3], end="\n\n\n")
#print(np.array(index2word(spa_t, vocab_idx2word_spa, vocab_word2idx_spa["[UNK]"]))[:3])

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size, layers, p_dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.dropout = nn.Dropout(p_dropout)

        self.gru = nn.GRU(embedding_size, hidden_size, layers, bidirectional=True, batch_first=True)

        self.l1 = nn.Linear(hidden_size * 2, hidden_size)
        self.l1_norm = nn.LayerNorm(hidden_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout(x)

        # output will have shape
        # (batch, seq_length, hidden_size * 2 if bidireccional else hidden_size)
        # hidden will have shape 
        # (layer * 2 if bidireccional else layer, batch_size, hidden_size)
        outputs, hidden = self.gru(x)

        # In order to compact the forward and backward of the hidden state.
        # This tensor will pass through a Linear layer using the last state
        # of layers concatenated. It'll have shape
        # (batch, hidden_size * 2)
        # instead of the original shape of the hidden state
        # (layer * 2 if bidireccional else layer, batch_size, hidden_size)
        x = torch.concat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        x = self.l1(x)
        x = self.l1_norm(x)
        x = torch.tanh(x)

        # To match expected dimentions in gru decoder
        x = torch.unsqueeze(x, dim=0)

        # x para decoder hidden state
        # output para cross-attention

        return outputs, hidden, x


class CrossAttention(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)


#class TeacherForcing(nn.Module):
#    def __init__(self, *args, **kwargs):
#        super().__init__(*args, **kwargs)


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size, layers, p_dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.gru = nn.GRU(embedding_size, hidden_size, layers, batch_first=True)
        self.dropout = nn.Dropout(p_dropout)

    def forward(self, x, decoder_hidden):
        
        x = self.embedding(x)
        x = self.dropout(x)

        outputs, hidden_state = self.gru(x, decoder_hidden)

        return outputs, hidden_state, x

In [ ]:
eng_batch, spa_o_batch, spa_t_batch = next(iter(train_loader))

encoder = Encoder(
    vocab_size=len(vocab_idx2word_eng), 
    embedding_size=300, 
    hidden_size=50,
    layers=1, 
    p_dropout=0.3
)
outputs_e, hidden_e, x_e = encoder(eng_batch)

print(eng_batch.shape)
print(outputs_e.shape)
print(hidden_e.shape)
print(x_e.shape)

torch.Size([64, 11])
torch.Size([64, 11, 100])
torch.Size([2, 64, 50])
torch.Size([1, 64, 50])


In [ ]:
decoder = Decoder(
    vocab_size=len(vocab_idx2word_spa),
    embedding_size=300,
    hidden_size=50, 
    layers=1,
    p_dropout=0.3
)
outputs_d, hidded_d, x_d = decoder(spa_o_batch, x_e)

tensor([[[ 0.5243, -2.2106, -0.1591,  ...,  0.8084,  0.7084, -0.7760],
         [-0.0996,  0.4497,  1.3873,  ..., -0.0000,  0.9042,  0.0000],
         [ 2.6296, -0.0000,  0.0000,  ...,  1.1439,  0.0000, -0.0000],
         ...,
         [ 0.4876,  2.1320,  0.2535,  ...,  0.5378, -2.7593,  1.2612],
         [ 0.0000,  2.1320,  0.0000,  ...,  0.5378, -2.7593,  1.2612],
         [ 0.4876,  2.1320,  0.0000,  ...,  0.5378, -2.7593,  0.0000]],

        [[ 0.5243, -0.0000, -0.1591,  ...,  0.0000,  0.0000, -0.7760],
         [-0.7248, -0.1879,  0.0000,  ...,  0.2579,  0.0000,  1.2192],
         [-1.6285,  0.6883,  0.2403,  ...,  0.9974, -0.0000, -2.0280],
         ...,
         [ 0.0000,  0.0000,  0.2535,  ...,  0.5378, -2.7593,  1.2612],
         [ 0.4876,  2.1320,  0.2535,  ...,  0.5378, -2.7593,  1.2612],
         [ 0.4876,  2.1320,  0.2535,  ...,  0.5378, -2.7593,  1.2612]],

        [[ 0.5243, -2.2106, -0.0000,  ...,  0.0000,  0.7084, -0.7760],
         [-0.8012,  2.9396,  0.0000,  ..., -1